# Fase 2 — Selección de algoritmo: el sistema contra el geométrico y el mono

Segunda fase de la narrativa nueva. Con el post-filtro **ya cerrado en la Fase 1**
(`smooth = s*`), se compara el sistema completo contra las alternativas:

- **DS** — geométrico robusto (WNG bajo, filtro estático).
- **MVDR-geo** — geométrico adaptativo (`MVDR_Recursive`), frágil al apuntado.
- **MVDR-geo+SCM-oracle** — el mismo MVDR geométrico pero con la covarianza de ruido
  **oracle**, tomada de la interferencia limpia multicanal (`MVDR_GEO_ORACLE_SCM`).
  Aísla el error del **modelo geométrico**: con estadísticas de ruido perfectas, lo que
  le siga faltando es culpa del steering vector y no de la estimación. Es la cota
  superior de la familia geométrica.
- **DTLN mono** — el baseline monocanal, que sale automático del motor en las
  columnas `dtln_alone_*` cuando hay intérpretes cargados. Corre con los modelos
  **float** (`model_{1,2}.tflite`): es el baseline en su mejor versión, sin
  restricciones de hardware.
- **DTLN mono int8** — el mismo mono con los modelos cuantizados
  (`model_quant_{1,2}.tflite`), los mismos que usa el sistema (`DTLN_MONO`).
- **NM-MVDR** — el sistema: `NM_MVDR_OFB_AUTO(smooth=s*, sharpen_exp=8, alpha=0.99)`,
  con el DTLN **int8** (`model_quant_1`), que es la configuración que se despliega en la
  Kria.
  La clase ya trae clavada la configuración medida (análisis rectangular + síntesis
  Hann, `block_update=1`, `leak=0`, `fuse='mean'`/`'ref'`, `pf_mask='isir'`), así que
  `smooth` es lo único que esta fase le pasa. **El nombre del algoritmo se mantiene**:
  en todas las tablas y figuras es `NM-MVDR`, como en el documento. `block_update=1`
  es el desacople temporal del cálculo de los pesos que hace realizable el lazo en
  hardware con baja latencia (ver la Fase 1) y, en este esquema, lo que permite que la
  red coma la **salida** del beamformer; con `=1` los pesos se siguen recalculando en
  todos los frames, sólo se retiene uno.

**`smooth` está clavado**, no se levanta de un JSON: `SMOOTH_STAR = 0.2`, el valor que
eligió la Fase 1 (corrida `F1_postfiltro_20260907_142125`) y que ahí resultó óptimo en
11 de las 12 celdas RT60 × iSIR. La búsqueda automática del `smooth_decision.json` sigue
en la celda de Config como camino alternativo —poner `SMOOTH_STAR = None` la reactiva—,
pero por defecto esta fase **no depende** de que la Fase 1 haya corrido en esta máquina.

**Qué cambia respecto de la Fase 1 original** (la que este notebook reemplaza):

- El único cambio en el diseño de la prueba es **qué objeto es `NM-MVDR`**: antes
  `NM_MVDR` (dos pasadas de DTLN, apuntado por DOA), ahora `NM_MVDR_OFB_AUTO`, el lazo
  cerrado **sobre la salida del beamformer** —sin front-end ni RTF—, con la máscara de
  las SCM fundida con la del canal de referencia y la del post-filtro agendada por un
  estimador **ciego** del iSIR. Las grillas, los procesadores geométricos y las figuras
  son las mismas.
- **Dos comparaciones contra el mono, dos afirmaciones distintas.**
  `NM-MVDR` vs `DTLN-mono-int8` es la **ablación controlada**: el mismo modelo de red en
  los dos brazos, cambia sólo el procesamiento espacial, y las limitaciones del DTLN
  (dominio DNS, poca reverberación) se cancelan. `NM-MVDR` vs `DTLN-mono` (float) es el
  **baseline más fuerte**: si el sistema int8 le gana igual, la conclusión no depende de
  haber cuantizado al rival. Antes la fase corría todo en int8.
- **Por qué el sistema queda en int8.** Es lo que se despliega, y cuantizarlo no le
  cuesta: `tests/dtln_quant_vs_float_check.py` (24 celdas MIRD) mide NM-MVDR float − int8
  = +0.01 ΔPESQ y +0.001 DNSMOS OVRL. Es estructural: el NM-MVDR usa sólo la etapa 1 del
  DTLN (la máscara), y la que sufre la cuantización es la etapa 2 (DNSMOS P808 +0.11 en
  23/24 celdas al des-cuantizarla), que usa únicamente el mono. Las Fases 1 y 3 corrieron
  con el sistema en int8, así que siguen siendo consistentes con esta.
- **Métricas no intrusivas** (`nonintrusive=True` en el motor): DNSMOS P.835
  (SIG/BAK/OVRL/P808) y TorchAudio-SQUIM (STOI/PESQ/SI-SDR estimados), al lado de las
  intrusivas, sobre la misma ventana y normalizadas a −26 dBFS RMS. En simulación hay
  ground truth, así que acá se **validan**: la sección final mide si ordenan los
  procesadores como las intrusivas. Es el puente para evaluar las grabaciones reales,
  donde sólo existen las no intrusivas.
- **`apply_dtln_post=False` en todo**: la cascada BF → DTLN quedó descartada del
  sistema, así que ya no se corre en ninguna prueba. `dtln_alone_*` (el mono solo)
  **sí** se calcula: es el baseline de la comparación.

La prueba:

- **P1 — alta diversidad acústica, sin errores.** Barrido amplio de RT60 × DOA-target
  × distancia × locutor × layout-interferentes × iSIR, con `error_*`/mismatch en 0.
  El geométrico recibe la posición **verdadera**, así que compite en igualdad → muestra
  que NM-MVDR al menos empata (y gana con reverberación) en todo el envelope. Con
  `MVDR-geo+SCM-oracle` adentro, la comparación además separa cuánto de la brecha del
  geométrico es **estimación de ruido** y cuánto es el **modelo geométrico** en sí.

**Qué se sacó de este notebook:** las pruebas **P2** (robustez a errores de sensor y
DOA) y **P3** (techos oracle: Oracle-mask / Oracle-SCM) ya no se corren acá. El
`MVDR-geo+SCM-oracle`, que vivía en P3, se mudó a P1 — que es donde la comparación de
algoritmo lo necesita. Las versiones anteriores del notebook quedan en el historial de
git si hiciera falta recuperarlas.

**Nota sobre la carga diagonal.** Se mantiene el criterio de la versión anterior:
todos calculan la carga igual (`load = max(rel·tr(Φ)/M, piso)`) pero **cada uno en su
óptimo medido** — el MVDR geométrico necesita carga pesada (`rel=1e-2`) para no
autocancelarse con un steering vector desajustado; la familia de máscara rinde mejor
con carga liviana. El ranking no cambia igualando los valores.

**Correr en orden:** Selector → Setup → Config → P1 → Figuras → No intrusivas.

## Selector de pruebas — qué correr en esta sesión

`RUN_P1` en `False` saltea la corrida y deja las celdas de figuras leyendo lo que haya
en memoria. Setup y Config se ejecutan siempre. Cada ejecución de la celda de Config
abre una carpeta de sesión nueva `F2_<RUN_ID>/`, así que lo corrido en esta sesión
queda junto y no pisa nada anterior.

In [ ]:
# ============== QUE PRUEBAS CORRER EN ESTA SESION ==============
# P1 = alta diversidad acustica:
#      DS / MVDR-geo / MVDR-geo+SCM-oracle / NM-MVDR / DTLN mono
RUN_P1 = True
# ===============================================================
print("a correr:", "P1" if RUN_P1 else "nada")

## Setup — ejecutar una vez por sesión de Colab
Montar Drive, clonar el repo, instalar dependencias y actualizar el código.

In [ ]:
# Import the drive module from Google Colab
from google.colab import drive

# Mount Google Drive to the virtual machine
drive.mount('/content/drive')

In [ ]:
# 3. Descargar tu código temporalmente
%cd /content
!git clone https://github.com/MatiasVereert/Vision-Aided-Beamformer.git

In [ ]:
import os, importlib, importlib.util

WHL = "/content/drive/MyDrive/colab_wheels"   # cache persistente de wheels en Drive
ip = get_ipython()

# (modulo que se importa, spec para instalar). git+ para las libs de GitHub;
# el resto por nombre. Mismos specs de siempre -> NO fuerza rebuild del cache.
PKGS = [
    ("noisereduce",     "noisereduce"),
    ("mir_eval",        "mir_eval"),
    ("pystoi",          "pystoi"),
    ("pesq",            "pesq"),
    ("paderbox",        "paderbox"),
    ("ai_edge_litert",  "ai_edge_litert"),
    # DNSMOS (metricas no intrusivas). SQUIM usa el torchaudio que trae Colab: NO
    # va en esta lista, para no arrastrar un torch distinto al del runtime.
    ("onnxruntime",     "onnxruntime"),
    ("librosa",         "librosa"),
    ("pb_bss",          "git+https://github.com/fgnt/pb_bss.git"),
    ("pyroomacoustics", "git+https://github.com/LCAV/pyroomacoustics.git"),
    ("nara_wpe",        "git+https://github.com/fgnt/nara_wpe.git"),
    ("fast_bss_eval",   "git+https://github.com/fakufaku/fast_bss_eval.git"),
]
def _name(spec):  # nombre instalable desde el cache (sin git+/.git)
    return spec.rsplit("/", 1)[-1].replace(".git", "") if spec.startswith("git+") else spec
BUILD = [spec for _, spec in PKGS]

# 1) (Re)construir el cache de wheels en Drive SOLO si cambio la lista (manifest).
manifest = os.path.join(WHL, ".manifest.txt")
key = "\n".join(sorted(BUILD))
if (not os.path.isfile(manifest)) or open(manifest).read() != key:
    os.makedirs(WHL, exist_ok=True)
    print("[*] (Re)construyendo cache de wheels en Drive (una vez por cambio de lista)...")
    ip.system(f"pip wheel --wheel-dir={WHL} " + " ".join(BUILD))
    with open(manifest, "w") as fh:
        fh.write(key)

# 2) Instalar desde el cache PAQUETE POR PAQUETE. Si a uno le falta la wheel
#    (p.ej. Colab cambio de version de Python), NO bloquea a los demas.
for mod, spec in PKGS:
    if importlib.util.find_spec(mod) is None:
        ip.system(f"pip install --no-index --find-links={WHL} {_name(spec)}")

# 3) AUTOCURA: lo que SIGA sin poder importarse se instala desde el indice
#    (PyPI/git) y se agrega al cache para la proxima sesion.
importlib.invalidate_caches()
missing = [(m, s) for m, s in PKGS if importlib.util.find_spec(m) is None]
if missing:
    print("[!] Faltan tras el cache:", [m for m, _ in missing], "-> instalando desde el indice...")
    ip.system("pip install " + " ".join(s for _, s in missing))
    ip.system(f"pip wheel --wheel-dir={WHL} " + " ".join(s for _, s in missing))
    importlib.invalidate_caches()

# 4) Verificacion final.
faltan = [m for m, _ in PKGS if importlib.util.find_spec(m) is None]
if faltan:
    print(f"[!] SIGUEN faltando {faltan}: reinicia el runtime "
          f"(Entorno de ejecucion > Reiniciar) y reejecuta esta celda.")
else:
    print("[*] Todas las dependencias OK.")
# Nuclear (si el cache quedo inservible tras un cambio de Python de Colab):
#   !rm -rf /content/drive/MyDrive/colab_wheels   y reejecuta -> reconstruye todo.

In [ ]:
%cd /content/Vision-Aided-Beamformer
!git pull origin main

## Config compartida (Fase 2)

In [ ]:
import sys, os, glob, json, numpy as np, shutil
from datetime import datetime

repo_root = '/content/Vision-Aided-Beamformer'
src_path = os.path.join(repo_root, 'src')
for p in (repo_root, src_path):
    if p not in sys.path: sys.path.append(p)
%cd {src_path}

try:
    import tensorflow as tf
    TFLITE_AVAILABLE = True
except ImportError:
    TFLITE_AVAILABLE = False

from evaluation.full_benchmark_test_dtln_mird import run_mird_grid_search
from evaluation.bf_wrappers import (DS, MVDR_Recursive, NM_MVDR_OFB_AUTO,
                                    MVDR_GEO_ORACLE_SCM, DTLN_MONO)
from propagation.mird_loader import MirdDatasetProvider

# MODELOS DEL DTLN.
#   NM-MVDR        -> int8 (m1_int8, via dtln_model_path): lo que se despliega en la
#                     Kria. Solo usa la etapa 1; cuantizarla le cuesta ~0.01 PESQ
#                     (tests/dtln_quant_vs_float_check.py).
#   DTLN-mono      -> float (m1 + m2, interpretes del motor -> dtln_alone_*): el
#                     baseline en su mejor version.
#   DTLN-mono-int8 -> int8 (m1_int8 + m2_int8, procesador DTLN_MONO): el mismo
#                     modelo que el sistema -> la ablacion controlada.
m1 = os.path.join(repo_root, "src/dnn_denoise/models/model_1.tflite")
m2 = os.path.join(repo_root, "src/dnn_denoise/models/model_2.tflite")
m1_int8 = os.path.join(repo_root, "src/dnn_denoise/models/model_quant_1.tflite")
m2_int8 = os.path.join(repo_root, "src/dnn_denoise/models/model_quant_2.tflite")
# El DTLN mono es UNO de los competidores de esta fase: los interpretes van SI o SI.
interpreter_1 = interpreter_2 = None
if TFLITE_AVAILABLE and os.path.exists(m1) and os.path.exists(m2):
    interpreter_1 = tf.lite.Interpreter(model_path=m1); interpreter_1.allocate_tensors()
    interpreter_2 = tf.lite.Interpreter(model_path=m2); interpreter_2.allocate_tensors()
    print("[*] DTLN TFLite float OK -> baseline mono disponible (dtln_alone_*).")
else:
    print("[!] Sin DTLN interpreters: NO va a haber baseline mono en esta corrida.")

# ================= RESULTADO DE LA FASE 1: el smooth elegido =================
# CLAVADO A MANO. 0.2 es lo que eligio la Fase 1 en la corrida
# F1_postfiltro_20260907_142125, donde ademas resulto ser el optimo en 11 de las 12
# celdas RT60 x iSIR (el schedule por iSIR y el schedule por RT60 colapsan los dos a
# la misma constante, ganancia 0.000 PESQ). Se clava para que esta fase NO dependa de
# que la Fase 1 haya corrido en esta maquina, y para no repetir la corrida en la que
# se colo un 0.0 por error.
#
# Poner SMOOTH_STAR = None reactiva la busqueda automatica del smooth_decision.json
# mas reciente en el Drive (el camino de abajo, que queda intacto).
SMOOTH_STAR = 0.2           # <<< None = leerlo de la Fase 1; o un float para forzarlo
_F1_GLOB = "/content/drive/MyDrive/Tesis_Beamformers/results_fases/F1_postfiltro_*/smooth_decision.json"
F1_DECISION = None
if SMOOTH_STAR is None:
    _cands = sorted(glob.glob(_F1_GLOB))
    if not _cands:
        raise RuntimeError(
            "No se encontro smooth_decision.json de la Fase 1.\n"
            f"Buscado en: {_F1_GLOB}\n"
            "Corre Fase_1_postfiltro_smooth.ipynb, o fija SMOOTH_STAR a mano.")
    F1_DECISION = json.load(open(_cands[-1]))
    SMOOTH_STAR = float(F1_DECISION["smooth_fijo"])
    print(f"[*] Fase 1: {os.path.dirname(_cands[-1])}")
    print(f"    smooth* = {SMOOTH_STAR:.2f} | schedule por iSIR recomendado: "
          f"{F1_DECISION.get('adaptativo_recomendado')}")
    if F1_DECISION.get("adaptativo_recomendado"):
        print("    [!] La Fase 1 recomendo PROGRAMAR el smooth con el iSIR. Esta fase "
              "corre con el valor FIJO (que es el caso conservador: el schedule solo "
              "puede mejorarlo). Ver smooth_por_isir en el JSON.")
else:
    print(f"[*] SMOOTH_STAR forzado a mano: {SMOOTH_STAR:.2f}")

# Configuracion FIJA del sistema (identica a la de la Fase 1). NM_MVDR_OFB_AUTO trae
# CLAVADO lo que se barrio antes (rect+hann, block_update=1, leak=0, fuse='mean'/'ref',
# pf_mask='isir' con la agenda en dB reales de iSIR), asi que aca solo quedan los dos
# escalares del nucleo de Souden y el smooth que fijo la Fase 1.
SYS = dict(sharpen_exp=8.0, alpha=0.99)

input_dir = "/content/drive/MyDrive/Benchmarks_tesis/inputs"
mird_dir  = "/content/drive/MyDrive/Benchmarks_tesis/rirs"
provider = MirdDatasetProvider(root_dir=mird_dir)

# ===================== PERILLAS =====================
DURATION = 15
# DNSMOS + SQUIM al lado de las intrusivas (~1 s por senal puntuada, ~9 s por
# celda). Si falta una dependencia, esas columnas salen en NaN y la corrida sigue.
NONINTRUSIVE = True
# ===================================================

TARGETS = [os.path.join(input_dir, f) for f in [
    "p002_emo_adoration_sentences.wav",
    "p008_emo_contentment_sentences.wav",
]]
INTERF = [os.path.join(input_dir, f) for f in [
    "techno_gated commune.wav",
    "hairdryer_07_SH_MKH800.wav",
    "drill_07_RHODE_NT1.wav",
]]

base_config = {
    'fs': 16000, 'duration': DURATION, 't_early': 0.050,
    'array_center': [3.0, 3.0, 1.2], 'mird_spacing': "3-3-3-8-3-3-3",
    'snr_db': 60.0,
    'source_path': TARGETS[0], 'interf_paths': INTERF,
    # --- WPE ELIMINADO DEL SISTEMA (use_wpe=False en toda grilla). Estos escalares
    #     solo existen porque el benchmark los exige en scene_base_config; no operan.
    'wpe_taps': 5, 'wpe_delay': 2, 'wpe_alpha': 0.9999,
    'wpe_stft_size': 512, 'wpe_stft_shift': 128,
    'stft_window': 512, 'stft_overlap': 384,
    'dtln_model_path': m1_int8,     # el SISTEMA corre en int8 (ver MODELOS arriba)
    'eval_references': ['early'],
}
assert 'dtln_sharpen_exp' not in base_config, "pisaria SYS['sharpen_exp']"

# --- PROCESADORES (Fase 2: la comparacion de algoritmo) ---
# DS = geometrico robusto (WNG bajo); MVDR-geo = geometrico adaptativo (fragil);
# NM-MVDR = EL SISTEMA (lazo sobre la salida + post-filtro de mascara ya calibrado).
# El DTLN mono float no va aca: lo calcula el motor solo (columnas dtln_alone_*).
# El int8 SI, como procesador: el motor tiene un solo par de interpretes. Con el
# int8 es la ablacion controlada (mismo modelo que el sistema); con el float, el
# baseline mas fuerte.
#
# CARGA DIAGONAL — los dos MVDR usan la MISMA formula,
#     load = max(carga_relativa * tr(Phi)/M , piso_absoluto),
# con VALORES DISTINTOS y a proposito: cada uno en el punto donde rinde mejor.
#   MVDR-geo -> rel = 1e-2 (carga PESADA). Sin regularizar, el desajuste del
#               steering vector en sala le hace cancelar el propio target.
#   NM-MVDR  -> el default de NM_MVDR_OFB_AUTO (min_loading=1e-9 en el nucleo de
#               Souden, que es el UNICO beamformer del esquema). Su informacion
#               espacial sale de los datos: puede cavar nulos profundos sin
#               lastimarse.
processors_dict = {
    "DS":       DS(),
    "MVDR-geo": MVDR_Recursive(rel_loading=1e-2, min_loading=1e-6),
    # d GEOMETRICO con Phi_NN ORACLE (de la interferencia limpia multicanal, sin VAD
    # ni mascara): la cota superior de la familia geometrica. Con estadisticas de
    # ruido perfectas, lo que le siga faltando contra NM-MVDR es el error del MODELO
    # geometrico -- la RTF de campo cercano en sala real--, no la estimacion. Misma
    # carga relativa que MVDR-geo (1e-2) para que la unica diferencia sea la Phi_NN.
    # Necesita 'oracle_noise' + geometria en la escena; los inyecta el motor.
    "MVDR-geo+SCM-oracle": MVDR_GEO_ORACLE_SCM(rel_loading=1e-2, alpha=0.99),
    "NM-MVDR":  NM_MVDR_OFB_AUTO(smooth=SMOOTH_STAR, **SYS),
    # El mono con el MISMO DTLN int8 que el sistema: la ablacion controlada.
    "DTLN-mono-int8": DTLN_MONO(m1_int8, m2_int8),
}

# Flag del selector: si se salteo esa celda, se corre P1.
globals().setdefault("RUN_P1", True)

# RUN_ID con SEGUNDOS -> no hay colisiones. TODAS las sub-corridas de ESTA
# ejecucion caen bajo UNA carpeta de sesion F2_<RUN_ID>/.
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
SESS_T = f"/content/results_temp/F2_{RUN_ID}"
SESS_D = f"/content/drive/MyDrive/Tesis_Beamformers/results_fases/F2_seleccion_{RUN_ID}"

def run(grid, name, procs=processors_dict):
    """Corre una sub-prueba. apply_dtln_post SIEMPRE False: la cascada BF->DTLN
    quedo descartada del sistema. El DTLN mono (dtln_alone_*) NO depende de ese
    flag: se calcula mientras haya interpretes."""
    t = os.path.join(SESS_T, name); d = os.path.join(SESS_D, name)
    os.makedirs(t, exist_ok=True); os.makedirs(d, exist_ok=True)
    df = run_mird_grid_search(grid_params=grid, dataset_provider=provider,
                              processors=procs, scene_base_config=base_config,
                              output_dir=t, interpreter_1=interpreter_1,
                              interpreter_2=interpreter_2, save_catalog=False,
                              apply_dtln_post=False, nonintrusive=NONINTRUSIVE)
    # DONE.json = marca de COMPLETITUD: se escribe SOLO si termino OK.
    n_proc = len(procs); n_cells = len(df) // max(n_proc, 1)
    json.dump({"run_id": RUN_ID, "phase": "F2", "experiment": name,
               "status": "COMPLETE", "rows": int(len(df)), "n_processors": n_proc,
               "n_cells": int(n_cells), "processors": list(procs.keys()),
               "dtln_post": False, "smooth_star": float(SMOOTH_STAR),
               "dtln_models": {"NM-MVDR": [os.path.basename(base_config['dtln_model_path'])],
                               "DTLN-mono": [os.path.basename(m1),
                                             os.path.basename(m2)],
                               "DTLN-mono-int8": [os.path.basename(m1_int8),
                                                  os.path.basename(m2_int8)]},
               "nonintrusive": bool(NONINTRUSIVE),
               # La clase VA EXPLICITA: SYS ya no la identifica (NM_MVDR_OFB_AUTO
               # trae clavado casi todo y de SYS solo quedan sharpen_exp/alpha), asi
               # que sin esto el DONE.json no dice que procesador corrio.
               "system": dict({"class": "NM_MVDR_OFB_AUTO"},
                              **{k: str(v) for k, v in SYS.items()}),
               "grid": {k: str(v) for k, v in grid.items()},
               "finished_utc": datetime.utcnow().isoformat() + "Z"},
              open(os.path.join(t, "DONE.json"), "w"), indent=2, ensure_ascii=False)
    shutil.copytree(t, d, dirs_exist_ok=True)
    print(f"[EXITO] {name} -> {d}  ({len(df)} filas, {n_proc} proc, {n_cells} celdas)")
    return df, d

# --------------------------------------------------------------------------
# BASELINE MONO como un "procesador" mas.
# El motor no lo devuelve como fila: lo deja en columnas dtln_alone_* REPETIDAS
# en todas las filas de una misma celda (una por procesador). Esta funcion
# deduplica por celda y lo apila con el nombre "DTLN-mono".
# Con use_wpe=False la senal 'wpe' ES el baseline, asi que Delta_dtln_alone_* ya
# es la mejora end-to-end del mono y se renombra a Delta_tot_* sin tocar nada.
# --------------------------------------------------------------------------
import pandas as pd

CELL_KEYS = ["rt60", "isir_db", "source", "target_angle", "target_dist",
             "interf_configs", "N_interferences", "error_angle_deg",
             "error_distance_m", "mismatch_gain", "mismatch_phase"]

def con_mono(df):
    cell = [c for c in CELL_KEYS if c in df.columns]
    cols = [c for c in df.columns if c.startswith("Delta_dtln_alone_")]
    if not cols:
        return df
    mono = df.drop_duplicates(subset=cell)[cell + cols].copy()
    mono = mono.rename(columns={c: c.replace("Delta_dtln_alone_", "Delta_tot_")
                                for c in cols})
    mono["processor"] = "DTLN-mono"
    return pd.concat([df, mono], ignore_index=True)

print(f"\nConfig Fase 2 lista. RUN_ID={RUN_ID}")
print(f"sesion -> {SESS_D}")
print("NM-MVDR =", f"NM_MVDR_OFB_AUTO(smooth={SMOOTH_STAR:.2f}, {SYS})")
print("procesadores P1 =", list(processors_dict.keys()), "+ DTLN mono float (automatico)")
print("DTLN: NM-MVDR", os.path.basename(base_config['dtln_model_path']),
      "| DTLN-mono", os.path.basename(m1), "+", os.path.basename(m2),
      "| DTLN-mono-int8", os.path.basename(m1_int8), "+", os.path.basename(m2_int8))
print("metricas no intrusivas:", NONINTRUSIVE)
print("flags:", {"RUN_P1": RUN_P1})

## P1 — alta diversidad acústica (resultado general, sin errores)

In [ ]:
if RUN_P1:
    # Barre RT60 x DOA-target x distancia x locutor x layout-interf x iSIR, con
    # mismatch y error_* en 0. Grilla marginalizada: recorta listas si tarda mucho.
    # MIRD disponible: RT60 {0.160,0.360,0.610}, dist {1,2} m, angulos +-90 en pasos de 15 deg.
    INTERF_P1 = [
        [(45, 1.0, 0)],                                   # 1 interferente
        [(30, 1.0, 0), (-45, 1.0, 1)],                    # 2 interferentes
        [(30, 1.0, 0), (-30, 1.0, 1), (60, 1.0, 2)],      # 3 interferentes
    ]
    grid_P1 = dict(
        rt60=[0.160, 0.360, 0.610],
        target_angle=[0, 30, 60],
        target_dist=[1.0, 2.0],
        source_path=TARGETS,
        interf_configs=INTERF_P1,
        isir_db=[-5, 0, 5, 10],
        use_wpe=[False],
        mismatch_gain=[0], mismatch_phase=[0],
        error_angle_deg=[0.0], error_distance_m=[0.0],
    )
    df_P1, dir_P1 = run(grid_P1, "P1_diversidad")
else:
    print("[i] P1 salteada (RUN_P1=False).")

### Preview rápido P1 (en memoria — incluye el baseline mono)

In [ ]:
if "df_P1" in globals():
    # Boxplots Delta por procesador sobre la diversidad de P1, con el DTLN mono
    # agregado como una "columna" mas a partir de las columnas dtln_alone_*.
    import numpy as np, matplotlib.pyplot as plt, seaborn as sns

    _d = con_mono(df_P1).replace([np.inf, -np.inf], np.nan)
    _pr = ["DS", "MVDR-geo", "MVDR-geo+SCM-oracle", "DTLN-mono-int8", "DTLN-mono",
           "NM-MVDR"]
    _pr = [p for p in _pr if p in set(_d.processor)]
    _c  = {"DS": "tab:green", "MVDR-geo": "tab:red",
           "MVDR-geo+SCM-oracle": "tab:purple",
           "DTLN-mono-int8": "lightsteelblue",
           "DTLN-mono": "tab:blue", "NM-MVDR": "tab:orange"}
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    for ax, (col, lbl) in zip(axes, [("Delta_tot_PESQ_early", "Δ PESQ"),
                                     ("Delta_tot_SIR_early", "Δ SIR [dB]")]):
        if col not in _d.columns: continue
        sns.boxplot(data=_d, x="processor", y=col, order=_pr, palette=_c, ax=ax, fliersize=1)
        ax.set_xlabel(""); ax.set_ylabel(lbl); ax.grid(alpha=0.3, axis="y")
        ax.tick_params(axis="x", rotation=15)
    plt.suptitle("Preview P1 — Δ por procesador (alta diversidad acústica)")
    plt.tight_layout(); plt.show()
else:
    print("[i] Preview P1: corré P1 en esta sesión para verlo.")

## Figuras — P1 (envelope general)

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
import seaborn as sns

METR  = [("Delta_tot_PESQ_early","Δ PESQ"), ("Delta_tot_STOI_early","Δ STOI"),
         ("Delta_tot_SDR_early","Δ SDR [dB]"), ("Delta_tot_SIR_early","Δ SIR [dB]")]
PROCS = ["DS", "MVDR-geo", "MVDR-geo+SCM-oracle", "DTLN-mono-int8", "DTLN-mono",
         "NM-MVDR"]
PAL   = {"DS":"tab:green", "MVDR-geo":"tab:red",
         "MVDR-geo+SCM-oracle":"tab:purple",
         "DTLN-mono-int8":"lightsteelblue",
         "DTLN-mono":"tab:blue", "NM-MVDR":"tab:orange"}

def con_mono(df):
    """Apila el baseline DTLN mono como un procesador mas (ver P1 preview)."""
    cell = [c for c in ["rt60", "isir_db", "source", "target_angle", "target_dist",
                        "interf_configs", "N_interferences", "error_angle_deg",
                        "error_distance_m", "mismatch_gain", "mismatch_phase"]
            if c in df.columns]
    cols = [c for c in df.columns if c.startswith("Delta_dtln_alone_")]
    if not cols:
        return df
    mono = df.drop_duplicates(subset=cell)[cell + cols].copy()
    mono = mono.rename(columns={c: c.replace("Delta_dtln_alone_", "Delta_tot_")
                                for c in cols})
    mono["processor"] = "DTLN-mono"
    return pd.concat([df, mono], ignore_index=True)

# --- P1: boxplots marginales por procesador (dispersion = incertidumbre real) ---
def fig_p1(dir_p1):
    dfp1 = con_mono(pd.read_csv(os.path.join(dir_p1, "mird_benchmark_metrics.csv"))) \
             .replace([np.inf, -np.inf], np.nan)
    order = [p for p in PROCS if p in set(dfp1.processor)]
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    for ax,(col,lbl) in zip(axes.ravel(), METR):
        if col not in dfp1.columns: continue
        sns.boxplot(data=dfp1, x="processor", y=col, order=order, palette=PAL, ax=ax, fliersize=1)
        ax.set_xlabel(""); ax.set_ylabel(lbl); ax.grid(alpha=0.3, axis="y")
        ax.tick_params(axis="x", rotation=15)
    fig.suptitle("Fase 2 · P1 — Δ sobre alta diversidad acústica")
    fig.tight_layout(); fig.savefig(os.path.join(dir_p1,"F2_P1_box.png"), dpi=140, bbox_inches="tight"); plt.show()

    # tendencia media vs RT60 (SDR/SIR)
    fig2, axes = plt.subplots(1, 2, figsize=(13, 5))
    for ax,(col,lbl) in zip(axes, METR[2:]):
        for pr in order:
            g = dfp1[dfp1.processor==pr].groupby("rt60")[col].mean()
            ax.plot(g.index.values*1000, g.values, "-o", color=PAL[pr], label=pr)
        ax.set_xlabel("RT60 [ms]"); ax.set_ylabel(lbl); ax.grid(alpha=0.3)
    axes[0].legend()
    fig2.suptitle("Fase 2 · P1 — tendencia vs RT60"); fig2.tight_layout()
    fig2.savefig(os.path.join(dir_p1,"F2_P1_rt.png"), dpi=140, bbox_inches="tight"); plt.show()

    # tendencia vs iSIR: es donde el mono y el espacial se separan
    fig3, axes = plt.subplots(1, 2, figsize=(13, 5))
    for ax,(col,lbl) in zip(axes, [METR[0], METR[3]]):
        for pr in order:
            g = dfp1[dfp1.processor==pr].groupby("isir_db")[col].median()
            ax.plot(g.index.values, g.values, "-o", color=PAL[pr], label=pr)
        ax.set_xlabel("iSIR [dB]"); ax.set_ylabel(lbl); ax.grid(alpha=0.3)
    axes[0].legend()
    fig3.suptitle("Fase 2 · P1 — tendencia vs iSIR (mediana)"); fig3.tight_layout()
    fig3.savefig(os.path.join(dir_p1,"F2_P1_isir.png"), dpi=140, bbox_inches="tight"); plt.show()

    # tabla resumen para el texto
    cols = [c for c,_ in METR if c in dfp1.columns]
    tab = dfp1.groupby("processor")[cols].median().rename(columns=dict(METR)).round(3)
    print("=== Fase 2 · P1 — Δ MEDIANO por procesador ===")
    print(tab.reindex(order).to_string())
    tab.to_csv(os.path.join(dir_p1, "F2_P1_resumen.csv"))

if "dir_P1" in globals():
    fig_p1(dir_P1)
else:
    print("[i] Corré P1 para las figuras de envelope general.")

## Métricas no intrusivas — validación contra las intrusivas

DNSMOS P.835 y SQUIM no necesitan la señal limpia, así que son las únicas que van a
existir sobre las grabaciones reales. Antes de usarlas ahí hay que saber **cuánto se
puede confiar en ellas**, y eso sólo se puede medir en simulación, donde están las dos
familias sobre las mismas señales.

La celda de abajo hace tres cosas:

1. **Boxplots y tabla de Δ no intrusivos por procesador** — el mismo envelope de P1
   visto con las métricas que no usan referencia.
2. **Correlación** (Spearman, sobre todas las filas celda × procesador de Δ) entre
   cada no intrusiva y la intrusiva que estima: ΔPESQ ↔ ΔDNSMOS OVRL / P808 / ΔSQUIM-PESQ,
   ΔSTOI ↔ ΔSQUIM-STOI, ΔSI-SDR ↔ ΔSQUIM-SI-SDR.
3. **Concordancia del ranking dentro de cada celda** — lo que de verdad importa para
   comparar sistemas sobre una grabación real: para cada par de procesadores de una
   misma escena, ¿la no intrusiva dice que gana el mismo que dice la intrusiva? Se
   reporta el % de pares concordantes, el Kendall τ mediano por celda, y aparte las
   comparaciones de la tesis (NM-MVDR contra cada rival).

**Sesgos conocidos, a leer con los números:** DNSMOS BAK premia la supresión agresiva
(favorece al mono); DNSMOS castiga la reverberación residual; y DNSMOS, SQUIM y el
DTLN salen todos del ecosistema DNS Challenge, así que pueden favorecer artefactos tipo
DTLN. Si el sistema gana igual en las no intrusivas, el resultado es más fuerte por eso
mismo.

In [ ]:
from scipy.stats import spearmanr, kendalltau

NI_METR = [("Delta_tot_DNSMOS_OVRL", "Δ DNSMOS OVRL"), ("Delta_tot_DNSMOS_SIG", "Δ DNSMOS SIG"),
           ("Delta_tot_DNSMOS_BAK", "Δ DNSMOS BAK"), ("Delta_tot_SQUIM_PESQ", "Δ SQUIM PESQ")]
# (intrusiva, no intrusiva) que miden lo mismo; las dos en Delta_tot_ (contra el mic de ref)
NI_PAIRS = [("PESQ_early", "DNSMOS_OVRL"), ("PESQ_early", "DNSMOS_P808"),
            ("PESQ_early", "SQUIM_PESQ"), ("STOI_early", "SQUIM_STOI"),
            ("SI-SDR_early", "SQUIM_SISDR")]

def fig_ni(dir_p1):
    dfp = con_mono(pd.read_csv(os.path.join(dir_p1, "mird_benchmark_metrics.csv"))) \
            .replace([np.inf, -np.inf], np.nan)
    if "Delta_tot_DNSMOS_OVRL" not in dfp.columns or dfp["Delta_tot_DNSMOS_OVRL"].isna().all():
        print("[i] Esta corrida no tiene metricas no intrusivas (NONINTRUSIVE=False o faltan deps).")
        return
    order = [p for p in PROCS if p in set(dfp.processor)]
    cell = [c for c in CELL_KEYS if c in dfp.columns]

    # --- 1. envelope no intrusivo ---
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    for ax, (col, lbl) in zip(axes.ravel(), NI_METR):
        if col not in dfp.columns or dfp[col].isna().all():
            ax.set_visible(False); continue
        sns.boxplot(data=dfp, x="processor", y=col, order=order, palette=PAL, ax=ax, fliersize=1)
        ax.set_xlabel(""); ax.set_ylabel(lbl); ax.grid(alpha=0.3, axis="y")
        ax.tick_params(axis="x", rotation=15)
    fig.suptitle("Fase 2 · P1 — Δ no intrusivos (sin referencia limpia)")
    fig.tight_layout(); fig.savefig(os.path.join(dir_p1, "F2_P1_ni_box.png"), dpi=140, bbox_inches="tight"); plt.show()

    ni_cols = [c for c in dfp.columns if c.startswith("Delta_tot_") and
               ("DNSMOS" in c or "SQUIM" in c) and dfp[c].notna().any()]
    tab = dfp.groupby("processor")[ni_cols].median().reindex(order)
    tab.columns = [c.replace("Delta_tot_", "Δ ") for c in tab.columns]
    print("=== Fase 2 · P1 — Δ no intrusivo MEDIANO por procesador ===")
    print(tab.round(3).to_string())
    tab.to_csv(os.path.join(dir_p1, "F2_P1_ni_resumen.csv"))

    # --- 2. correlacion intrusiva <-> no intrusiva, sobre todas las filas ---
    rows = []
    for intr, ni in NI_PAIRS:
        a, b = f"Delta_tot_{intr}", f"Delta_tot_{ni}"
        if a not in dfp.columns or b not in dfp.columns:
            continue
        ok = dfp[[a, b]].dropna()
        if len(ok) < 3:
            continue
        rho = spearmanr(ok[a], ok[b]).correlation

        # --- 3. concordancia del ranking DENTRO de cada celda ---
        pa = dfp.pivot_table(index=cell, columns="processor", values=a)
        pb = dfp.pivot_table(index=cell, columns="processor", values=b)
        procs = [p for p in order if p in pa.columns and p in pb.columns]
        agree = tot = 0
        for i, p in enumerate(procs):
            for q in procs[i + 1:]:
                da, db = pa[p] - pa[q], pb[p] - pb[q]
                m = da.notna() & db.notna() & (da != 0) & (db != 0)
                agree += int((np.sign(da[m]) == np.sign(db[m])).sum()); tot += int(m.sum())
        taus = [kendalltau(pa.loc[k, procs], pb.loc[k, procs], nan_policy="omit").statistic
                for k in pa.index]
        rows.append({"intrusiva": intr, "no_intrusiva": ni, "n_filas": len(ok),
                     "spearman": rho, "pares_concordantes_%": 100 * agree / max(tot, 1),
                     "kendall_tau_mediano_celda": np.nanmedian(taus)})
    val = pd.DataFrame(rows)
    print("\n=== Validacion: ¿la no intrusiva ordena como la intrusiva? ===")
    print(val.round(3).to_string(index=False))

    # Las comparaciones de la tesis: NM-MVDR contra cada rival, celda por celda.
    if "NM-MVDR" in order:
        print("\n=== NM-MVDR gana en ... celdas (intrusiva | no intrusiva | concuerdan) ===")
        comp = []
        for intr, ni in NI_PAIRS:
            a, b = f"Delta_tot_{intr}", f"Delta_tot_{ni}"
            if a not in dfp.columns or b not in dfp.columns:
                continue
            pa = dfp.pivot_table(index=cell, columns="processor", values=a)
            pb = dfp.pivot_table(index=cell, columns="processor", values=b)
            for rival in [p for p in order if p != "NM-MVDR"]:
                if rival not in pa.columns or rival not in pb.columns:
                    continue
                wa, wb = pa["NM-MVDR"] > pa[rival], pb["NM-MVDR"] > pb[rival]
                m = (pa["NM-MVDR"] - pa[rival]).notna() & (pb["NM-MVDR"] - pb[rival]).notna()
                comp.append({"par": f"{intr} / {ni}", "rival": rival, "n": int(m.sum()),
                             "gana_intrusiva": int(wa[m].sum()), "gana_no_intrusiva": int(wb[m].sum()),
                             "concuerdan": int((wa[m] == wb[m]).sum())})
        comp = pd.DataFrame(comp)
        print(comp.to_string(index=False))
        comp.to_csv(os.path.join(dir_p1, "F2_P1_ni_vs_rivales.csv"), index=False)
    val.to_csv(os.path.join(dir_p1, "F2_P1_ni_validacion.csv"), index=False)

    # dispersion de la mas usada: ΔPESQ vs ΔDNSMOS OVRL, coloreado por procesador
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    for ax, (intr, ni) in zip(axes, [("PESQ_early", "DNSMOS_OVRL"), ("PESQ_early", "SQUIM_PESQ")]):
        a, b = f"Delta_tot_{intr}", f"Delta_tot_{ni}"
        if a not in dfp.columns or b not in dfp.columns or dfp[b].isna().all():
            ax.set_visible(False); continue
        for pr in order:
            d = dfp[dfp.processor == pr]
            ax.scatter(d[a], d[b], s=12, alpha=0.6, color=PAL[pr], label=pr)
        ax.set_xlabel(f"Δ {intr.replace('_early', '')} (intrusiva)")
        ax.set_ylabel(f"Δ {ni} (no intrusiva)"); ax.grid(alpha=0.3)
    axes[0].legend(fontsize=8)
    fig.suptitle("Fase 2 · P1 — intrusiva vs no intrusiva (una fila por celda × procesador)")
    fig.tight_layout(); fig.savefig(os.path.join(dir_p1, "F2_P1_ni_scatter.png"), dpi=140, bbox_inches="tight"); plt.show()

if "dir_P1" in globals():
    fig_ni(dir_P1)
else:
    print("[i] Corré P1 para la validacion de las metricas no intrusivas.")